# Reduced-Time Recursive Physics-Informed Neural Network for Shape Memory Materials

## Uniaxial Tension and Shape-Memory Recovery Cycle

This notebook implements a physics-informed neural network for thermo-viscoelastic shape memory materials, used to predict the shape memory recovery behavior of 45° fiber-reinforced composite materials under uniaxial tensile loading.

### Problem Setup

- **Geometry**: Rectangular solid block (33mm × 6mm × 2mm)
- **Boundary Conditions**: Left face (x=0) fixed, right face (x=33mm) with prescribed displacement loading
- **Loading History**: Shape memory cycle (programming → cooling → unloading → reheating)

## Import Required Libraries

Import libraries for numerical computation, deep learning, data processing, and visualization.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
import time
from pathlib import Path
from matplotlib.tri import Triangulation
from scipy.interpolate import interp1d

## Set Random Seeds and Device

Set random seeds for reproducibility. Detect and configure computing device (GPU or CPU).

In [2]:
torch.manual_seed(42)
np.random.seed(42)

# Check for GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


## FEDataLoader

Implementation of the `FEDataLoader` class.

Load and process FE simulation results from CSV files

In [ ]:
class FEDataLoader:

    def __init__(self, data_dir, step_frame_time_file=None):
        self.data_dir = Path(data_dir)
        self.step_frame_time_file = Path(step_frame_time_file) if step_frame_time_file else None
        self.data = []
        self.step_frame_time_map = None
        self.steps_info = {
            1: {'duration': 20.0, 'T_start': 343.0, 'T_end': 343.0},
            2: {'duration': 50.0, 'T_start': 343.0, 'T_end': 298.0},
            3: {'duration': 1.0, 'T_start': 298.0, 'T_end': 298.0},
            4: {'duration': 50.0, 'T_start': 298.0, 'T_end': 353.0}
        }
        self.cumulative_time = {1: 0.0, 2: 20.0, 3: 70.0, 4: 71.0}

    def load_all_data(self):
        if self.step_frame_time_file and self.step_frame_time_file.exists():
            sft_df = pd.read_csv(self.step_frame_time_file, header=None,
                                names=['Step', 'Frame', 'Time'])
            self.step_frame_time_map = {
                (row['Step'], row['Frame']): row['Time']
                for _, row in sft_df.iterrows()
            }
        else:
            self.step_frame_time_map = None

        if not self.data_dir.exists():
            raise FileNotFoundError(f"Data directory not found: {self.data_dir}")

        for step in range(1, 5):
            step_files = sorted(
                list(self.data_dir.glob(f"*_Step-{step}_frame*.csv")),
                key=lambda x: int(x.stem.split('frame')[-1])
            )

            if len(step_files) == 0:
                continue

            for csv_file in step_files:
                df = pd.read_csv(csv_file)
                frame_idx = int(csv_file.stem.split('frame')[-1])

                if self.step_frame_time_map is not None:
                    if (step, frame_idx) in self.step_frame_time_map:
                        frame_time = self.step_frame_time_map[(step, frame_idx)]
                    else:
                        raise ValueError(f"Step {step}, Frame {frame_idx} not found in step-frame-time mapping!")
                else:
                    step_duration = self.steps_info[step]['duration']
                    num_frames = len(step_files) - 1
                    if num_frames > 0:
                        frame_time = self.cumulative_time[step] + (frame_idx / num_frames) * step_duration
                    else:
                        frame_time = self.cumulative_time[step]

                T_start = self.steps_info[step]['T_start']
                T_end = self.steps_info[step]['T_end']
                step_duration = self.steps_info[step]['duration']

                if step_duration > 0 and T_start != T_end:
                    time_in_step = frame_time - self.cumulative_time[step]
                    time_fraction = time_in_step / step_duration
                    temperature = T_start + (T_end - T_start) * time_fraction
                else:
                    temperature = T_start

                df['Time'] = frame_time
                df['Temperature'] = temperature
                df['Step'] = step
                df['Frame'] = frame_idx

                self.data.append(df)

        if len(self.data) == 0:
            raise ValueError(f"No CSV files found in {self.data_dir}. Please check the path and file names.")

        self.full_data = pd.concat(self.data, ignore_index=True)

        return self.full_data

    def get_domain_bounds(self):
        bounds = {
            'x_min': self.full_data['X'].min(),
            'x_max': self.full_data['X'].max(),
            'y_min': self.full_data['Y'].min(),
            'y_max': self.full_data['Y'].max(),
            'z_min': self.full_data['Z'].min(),
            'z_max': self.full_data['Z'].max(),
            't_min': self.full_data['Time'].min(),
            't_max': self.full_data['Time'].max(),
        }
        return bounds

    def get_boundary_nodes(self):
        x_min = self.full_data['X'].min()
        left_nodes = self.full_data[self.full_data['X'] == x_min]
        return left_nodes

## MaterialParameters

Implementation of the `MaterialParameters` class.

In [4]:
class MaterialParameters:

    def __init__(self):
        # Reference temperature from UMAT (line 72: Tr=323)
        self.T_ref = 323.0  # K (50°C)

        # Switching temperature from UMAT (line 102: 317.4 K)
        self.T_switch = 317.4  # K (44.25°C)

        # WLF parameters from UMAT (lines 73-74)
        self.C1 = 14.8
        self.C2 = 45.6

        # Arrhenius parameters from UMAT (line 105)
        self.E_arrhenius = 27403.3  # Arrhenius coefficient
        self.T_arr_ref = 336.0  # K (reference for Arrhenius)

        # Read from: *User Material, constants=43

        # PROPS(1-6): Relaxation times rhoi (seconds)
        self.rho = np.array([0.1, 1.0, 10.0, 100.0, 1000.0, 10000.0])

        # PROPS(7) + PROPS(8-13): C11inf and C11i
        self.C11_inf = 10319.1  # MPa
        self.C11_prony = np.array([200.254, 89.0021, 288.039, 322.882, 29.7953, 1.08799])  # MPa

        # PROPS(14) + PROPS(15-20): C12inf and C12i
        self.C12_inf = 1.77138  # MPa
        self.C12_prony = np.array([183.83, 82.8455, 272.998, 318.639, 30.0832, 1.0973])  # MPa

        # PROPS(21) + PROPS(22-27): C22inf and C22i
        self.C22_inf = 2.83844  # MPa
        self.C22_prony = np.array([293.707, 132.426, 436.637, 510.272, 48.2092, 1.75608])  # MPa

        # PROPS(28) + PROPS(29-34): C23inf and C23i
        self.C23_inf = 1.82402  # MPa
        self.C23_prony = np.array([188.472, 84.9984, 280.342, 327.818, 30.9813, 1.1279])  # MPa

        # PROPS(35) + PROPS(36-41): C66inf and C66i
        self.C66_inf = 0.530688  # MPa
        self.C66_prony = np.array([55.3091, 24.9132, 82.0325, 95.5597, 9.01153, 0.328643])  # MPa

        # PROPS(42-43): Thermal expansion coefficients Tp
        Tp1 = 3.7e-6   # PROPS(42)
        Tp2 = 0.00012  # PROPS(43)
        self.Tp = np.array([Tp1, Tp2, Tp2, 0.0, 0.0, 0.0])  # Tp(1), Tp(2)=Tp(3), rest=0

        # Convert to Pa (multiply by 1e6)
        self.C11_inf *= 1e6
        self.C11_prony *= 1e6
        self.C12_inf *= 1e6
        self.C12_prony *= 1e6
        self.C22_inf *= 1e6
        self.C22_prony *= 1e6
        self.C23_inf *= 1e6
        self.C23_prony *= 1e6
        self.C66_inf *= 1e6
        self.C66_prony *= 1e6

        # Prony series parameters
        self.N_prony = 6

        # Use characteristic length (coupon length) and equilibrium modulus
        self.L_ref = 33.0  # mm (characteristic length: coupon length along X)
        self.E_ref = self.C11_inf  # Pa (use actual C11_inf as reference modulus)

        # C_inf: Equilibrium stiffness matrix (6x6 Voigt notation)
        # For orthotropic material (assuming simplified 3D version)
        self.C_inf = np.zeros((6, 6))
        # Normal stress-strain coupling
        self.C_inf[0, 0] = self.C11_inf  # σ11 = C11*ε11 + ...
        self.C_inf[0, 1] = self.C12_inf  # σ11 = ... + C12*ε22 + ...
        self.C_inf[0, 2] = self.C12_inf  # σ11 = ... + C12*ε33
        self.C_inf[1, 0] = self.C12_inf  # σ22 = C12*ε11 + ...
        self.C_inf[1, 1] = self.C22_inf  # σ22 = ... + C22*ε22 + ...
        self.C_inf[1, 2] = self.C23_inf  # σ22 = ... + C23*ε33
        self.C_inf[2, 0] = self.C12_inf  # σ33 = C12*ε11 + ...
        self.C_inf[2, 1] = self.C23_inf  # σ33 = ... + C23*ε22 + ...
        self.C_inf[2, 2] = self.C22_inf  # σ33 = ... + C22*ε33
        # Shear moduli
        self.C_inf[3, 3] = (self.C22_inf - self.C23_inf) / 2.0  # G23
        self.C_inf[4, 4] = self.C66_inf  # G13
        self.C_inf[5, 5] = self.C66_inf  # G12

        # Prony series stiffness matrices (list of 6x6 matrices, one per branch)
        self.C_prony = []
        for i in range(self.N_prony):
            C_i = np.zeros((6, 6))
            C_i[0, 0] = self.C11_prony[i]
            C_i[0, 1] = self.C12_prony[i]
            C_i[0, 2] = self.C12_prony[i]
            C_i[1, 0] = self.C12_prony[i]
            C_i[1, 1] = self.C22_prony[i]
            C_i[1, 2] = self.C23_prony[i]
            C_i[2, 0] = self.C12_prony[i]
            C_i[2, 1] = self.C23_prony[i]
            C_i[2, 2] = self.C22_prony[i]
            C_i[3, 3] = (self.C22_prony[i] - self.C23_prony[i]) / 2.0
            C_i[4, 4] = self.C66_prony[i]
            C_i[5, 5] = self.C66_prony[i]
            self.C_prony.append(C_i)

        # Relaxation times (tau_0 = rho, same as UMAT)
        self.tau_0 = self.rho

    def shift_factor(self, T):
        """
        Calculate time-temperature shift factor a_T(T) matching UMAT lines 102-106
        WLF: aT = 10^(-c1*(T-Tr)/(c2+T-Tr)) for T > 317.4 K
        Arrhenius: aT = exp(27403.3*(1/T - 1/336.0)) for T <= 317.4 K
        """
        if isinstance(T, torch.Tensor):
            T = torch.clamp(T, min=250.0, max=400.0)  # Avoid numerical issues
            # WLF above T_switch
            log10_aT_wlf = -self.C1 * (T - self.T_ref) / (self.C2 + T - self.T_ref)
            aT_wlf = 10.0 ** log10_aT_wlf
            # Arrhenius below T_switch
            aT_arr = torch.exp(self.E_arrhenius * (1.0/T - 1.0/self.T_arr_ref))
            # Select based on temperature
            a_T = torch.where(T > self.T_switch, aT_wlf, aT_arr)
        else:
            T = np.clip(T, 250.0, 400.0)
            if T > self.T_switch:
                # WLF
                log10_aT = -self.C1 * (T - self.T_ref) / (self.C2 + T - self.T_ref)
                a_T = 10.0 ** log10_aT
            else:
                # Arrhenius
                a_T = np.exp(self.E_arrhenius * (1.0/T - 1.0/self.T_arr_ref))
        return a_T

## RT-RPINN

Implementation of the RT-RPINN model class.

RT-RPINN for shape memory materials with internal variables

In [5]:
class SpatiotemporalPINN(nn.Module):
    """
    PINN for shape memory materials with internal variables
    Input: (x, y, z, t, T) - spatial coordinates, time, and temperature
    Output: (u_x, u_y, u_z) + q[6,6] hereditary variables
    Note: Temperature T is now explicitly passed as input (5th dimension)
    """

    def __init__(self, layers=[5, 128, 128, 128, 128], n_prony=6, output_internal_vars=True):
        super(SpatiotemporalPINN, self).__init__()

        self.n_prony = n_prony
        self.output_internal_vars = output_internal_vars

        # Shared layers for feature extraction
        self.shared_layers = nn.ModuleList()
        for i in range(len(layers) - 1):
            self.shared_layers.append(nn.Linear(layers[i], layers[i+1]))

        # Output layer for displacement (3 components)
        self.displacement_head = nn.Linear(layers[-2], 3)

        # Output layers for internal variables q (6 strain components × n_prony branches)
        if output_internal_vars:
            self.internal_var_head = nn.Linear(layers[-2], 6 * n_prony)

        # Initialize weights
        for layer in self.shared_layers:
            nn.init.xavier_normal_(layer.weight)
            nn.init.zeros_(layer.bias)
        nn.init.xavier_normal_(self.displacement_head.weight)
        nn.init.zeros_(self.displacement_head.bias)
        if output_internal_vars:
            nn.init.xavier_normal_(self.internal_var_head.weight)
            nn.init.zeros_(self.internal_var_head.bias)

    def forward(self, x, y, z, t, T):
        """Forward pass through the network"""
        inputs = torch.cat([x, y, z, t, T], dim=1)

        # Shared feature extraction
        features = inputs
        for i in range(len(self.shared_layers)):
            features = torch.tanh(self.shared_layers[i](features))

        # Displacement output
        u = self.displacement_head(features)
        u_x, u_y, u_z = u[:, 0:1], u[:, 1:2], u[:, 2:3]

        if self.output_internal_vars:
            # Internal variables output q (shape: batch_size, 6*n_prony)
            q_flat = self.internal_var_head(features)
            # Reshape to (batch_size, 6, n_prony)
            q = q_flat.reshape(-1, 6, self.n_prony)
            return u_x, u_y, u_z, q
        else:
            return u_x, u_y, u_z

## Theoretical Background

### Constitutive Model

The thermo-viscoelastic constitutive model incorporates:

1. **Coordinate Transformation**: Material fiber direction is rotated 45° in XY plane from global X-axis. The transformation between global and material frames is:
   - Strain: $\varepsilon_{material} = Q \cdot \varepsilon_{global} \cdot Q^T$
   - Stress: $\sigma_{global} = Q^T \cdot \sigma_{material} \cdot Q$

2. **Prony Series Viscoelasticity**: The stress response includes equilibrium and viscoelastic contributions:
   $$\sigma = C_{\infty} : \varepsilon_{mech} + \sum_{i=1}^{N} C_i : (\varepsilon_{mech} - q_i)$$

3. **Internal Variables Evolution**: The internal variables $q_i$ evolve according to:
   $$q_i^{n+1} = \gamma_i \cdot q_i^n + (1 - \gamma_i) \cdot \varepsilon_{mech}^{n+1}$$
   where $\gamma_i = \exp(-\Delta t / (a_T \cdot \tau_i))$ and $a_T$ is the time-temperature shift factor.

4. **Time-Temperature Superposition**: The shift factor $a_T(T)$ follows:
   - WLF equation for $T > T_{switch}$: $\log_{10}(a_T) = -C_1(T-T_r)/(C_2+T-T_r)$
   - Arrhenius equation for $T \leq T_{switch}$: $a_T = \exp(E_a(1/T - 1/T_{ref}))$

5. **Thermal Strain**: Thermal expansion is modeled as $\varepsilon_{th} = T_p \cdot (T - T_{ref})$ where $T_p$ is the thermal expansion coefficient vector.

### Governing Equations

The momentum balance equation (assuming zero body forces):
$$\nabla \cdot \sigma = 0$$

The small strain assumption: $\varepsilon = \frac{1}{2}(\nabla u + \nabla u^T)$

## RT-RPINN Solver

Implementation of the RT-RPINN solver.

RT-RPINN solver for thermo-viscoelastic shape memory materials with FE data

In [ ]:
class PINNSolver:
    """
    PINN solver for thermo-viscoelastic shape memory materials with FE data
    """

    def __init__(self, model, material_params, fe_data_loader=None, bounds=None, use_physics_loss=True):
        self.model = model.to(device)
        self.mat = material_params
        self.fe_loader = fe_data_loader
        self.bounds = bounds
        self.use_physics_loss = use_physics_loss

        self.L_ref = material_params.L_ref  # mm (geometric length scale)
        self.E_ref = material_params.E_ref  # Pa (material stiffness scale, C11_inf)

        self.lambda_data = 1.0  # Data loss (displacements in mm)
        self.lambda_pde = 0.7 if use_physics_loss else 0.0  # PDE loss (matching formal training script)
        self.lambda_bc = 1.0  # BC loss (displacements in mm)
        self.lambda_internal = 1.0 if use_physics_loss else 0.0  # Internal variable consistency

        # Adaptive weights for displacement components to balance different magnitudes
        # U1 (main loading): ~5mm range
        # U2 (coupling): ~0.8mm range (~1/6 of U1)
        # U3 (Poisson): ~0.18mm range (~1/28 of U1)
        self.lambda_u1 = 1.0   # Base weight for main loading direction
        self.lambda_u2 = 10.0   # Matching formal training script
        self.lambda_u3 = 300.0  # Matching formal training script

        # Training history
        self.loss_history = []
        self.loss_components_history = []

        # Transformation: eps_material = Q * eps_global * Q^T (for tensor form)
        # For stress: sigma_global = Q^T * sigma_material * Q
        theta = np.deg2rad(45.0)
        c, s = np.cos(theta), np.sin(theta)

        self.Q = torch.tensor([
            [ c,  s,  0],
            [-s,  c,  0],
            [ 0,  0,  1]
        ], dtype=torch.float32, device=device)

        # Add evolution loss weight
        # New discrete form is much more stable (no time derivatives!)
        # Residual is in "strain" units, naturally same scale as data loss
        self.lambda_evol = 0.1 if use_physics_loss else 0.0  # Internal variable evolution (discrete form)

    def normalize_residual(self, residual, scale):
        """
        Normalize residual by reference scale to make loss unitless
        Args:
            residual: tensor of residuals
            scale: reference magnitude for this physical quantity
        Returns:
            normalized squared residual
        """
        return torch.mean((residual / scale) ** 2)

    def get_temperature(self, t):
        """Calculate temperature based on time according to simulation steps"""
        # t is a tensor
        T = torch.zeros_like(t)
        mask1 = t < 20.0
        T[mask1] = 343.0
        mask2 = (t >= 20.0) & (t < 70.0)
        T[mask2] = 343.0 - (343.0 - 298.0) * (t[mask2] - 20.0) / 50.0
        mask3 = (t >= 70.0) & (t < 71.0)
        T[mask3] = 298.0
        mask4 = t >= 71.0
        T[mask4] = 298.0 + (353.0 - 298.0) * (t[mask4] - 71.0) / 50.0
        return T

    def compute_strain(self, u_x, u_y, u_z, x, y, z):
        """
        Compute strain components from displacement
        Small strain assumption: ε = 1/2(∇u + ∇u^T)
        """
        # Compute gradients
        u_x_x = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u_x),
                                     create_graph=True)[0]
        u_y_y = torch.autograd.grad(u_y, y, grad_outputs=torch.ones_like(u_y),
                                     create_graph=True)[0]
        u_z_z = torch.autograd.grad(u_z, z, grad_outputs=torch.ones_like(u_z),
                                     create_graph=True)[0]

        u_x_y = torch.autograd.grad(u_x, y, grad_outputs=torch.ones_like(u_x),
                                     create_graph=True)[0]
        u_y_x = torch.autograd.grad(u_y, x, grad_outputs=torch.ones_like(u_y),
                                     create_graph=True)[0]

        u_x_z = torch.autograd.grad(u_x, z, grad_outputs=torch.ones_like(u_x),
                                     create_graph=True)[0]
        u_z_x = torch.autograd.grad(u_z, x, grad_outputs=torch.ones_like(u_z),
                                     create_graph=True)[0]

        u_y_z = torch.autograd.grad(u_y, z, grad_outputs=torch.ones_like(u_y),
                                     create_graph=True)[0]
        u_z_y = torch.autograd.grad(u_z, y, grad_outputs=torch.ones_like(u_z),
                                     create_graph=True)[0]

        # Strain components (Voigt notation: ε11, ε22, ε33, γ23, γ13, γ12)
        eps_11 = u_x_x
        eps_22 = u_y_y
        eps_33 = u_z_z
        gamma_23 = u_y_z + u_z_y
        gamma_13 = u_x_z + u_z_x
        gamma_12 = u_x_y + u_y_x

        return eps_11, eps_22, eps_33, gamma_23, gamma_13, gamma_12

    def compute_thermal_strain(self, T, T_old=None):
        """
        Compute thermal strain components using Tp vector from UMAT
        Thermal strain = Tp * (T - T_ref)
        """
        if T_old is None:
            T_old = self.mat.T_ref
        dT = T - T_old
        # Tp has 6 components for Voigt notation
        Tp_tensor = torch.from_numpy(self.mat.Tp).float().to(device)
        eps_th = Tp_tensor.unsqueeze(0) * dT  # Shape: (batch, 6)
        return eps_th[:, 0:1], eps_th[:, 1:2], eps_th[:, 2:3], eps_th[:, 3:4], eps_th[:, 4:5], eps_th[:, 5:6]

    def compute_stress(self, eps_11, eps_22, eps_33, gamma_23, gamma_13, gamma_12,
                      eps_th_11, eps_th_22, eps_th_33, eps_th_23, eps_th_13, eps_th_12,
                      T, t, q=None):
        """
        Compute stress using FULL thermo-viscoelastic constitutive model with:
        1. 45° coordinate transformation (global -> material frame)
        2. Prony series viscoelasticity (6 branches)
        3. Internal variables q (shape memory effect)

        - Global strain -> Rotate to material frame -> Subtract thermal strain
        - Material stress = C_inf : eps_mech + Σ_i C_i : (eps_mech - q_i)
        - Material stress -> Rotate back to global frame

        Args:
            eps_*, gamma_*: Global frame strains (Voigt notation)
            eps_th_*: Thermal strains in material frame
            q: Internal variables (batch, 6, N_prony) in material frame, or None

        Returns:
            sigma_* (global frame), eps_mech_* (material frame for evolution equation)
        """
        batch_size = eps_11.shape[0]

        eps_glob_tensor = torch.zeros(batch_size, 3, 3, device=device)
        eps_glob_tensor[:, 0, 0] = eps_11.squeeze()
        eps_glob_tensor[:, 1, 1] = eps_22.squeeze()
        eps_glob_tensor[:, 2, 2] = eps_33.squeeze()
        eps_glob_tensor[:, 0, 1] = eps_glob_tensor[:, 1, 0] = gamma_12.squeeze() / 2.0
        eps_glob_tensor[:, 0, 2] = eps_glob_tensor[:, 2, 0] = gamma_13.squeeze() / 2.0
        eps_glob_tensor[:, 1, 2] = eps_glob_tensor[:, 2, 1] = gamma_23.squeeze() / 2.0

        # eps_mat = Q * eps_glob * Q^T
        Q = self.Q  # (3, 3)
        eps_mat_tensor = torch.matmul(torch.matmul(Q, eps_glob_tensor), Q.T)

        em_11 = eps_mat_tensor[:, 0, 0:1]
        em_22 = eps_mat_tensor[:, 1, 1:2]
        em_33 = eps_mat_tensor[:, 2, 2:3]
        gm_23 = 2.0 * eps_mat_tensor[:, 1, 2:3]
        gm_13 = 2.0 * eps_mat_tensor[:, 0, 2:3]
        gm_12 = 2.0 * eps_mat_tensor[:, 0, 1:2]

        em_mech_11 = em_11 - eps_th_11
        em_mech_22 = em_22 - eps_th_22
        em_mech_33 = em_33 - eps_th_33
        gm_mech_23 = gm_23 - eps_th_23
        gm_mech_13 = gm_13 - eps_th_13
        gm_mech_12 = gm_12 - eps_th_12

        # Equilibrium part: sigma_eq = C_inf : eps_mech
        C11_inf = self.mat.C11_inf
        C12_inf = self.mat.C12_inf
        C22_inf = self.mat.C22_inf
        C23_inf = self.mat.C23_inf
        C66_inf = self.mat.C66_inf

        sm_11 = C11_inf * em_mech_11 + C12_inf * em_mech_22 + C12_inf * em_mech_33
        sm_22 = C12_inf * em_mech_11 + C22_inf * em_mech_22 + C23_inf * em_mech_33
        sm_33 = C12_inf * em_mech_11 + C23_inf * em_mech_22 + C22_inf * em_mech_33
        sm_23 = ((C22_inf - C23_inf) / 2.0) * gm_mech_23
        sm_13 = C66_inf * gm_mech_13
        sm_12 = C66_inf * gm_mech_12

        if q is not None and self.use_physics_loss:
            for i in range(self.mat.N_prony):
                C11_i = self.mat.C11_prony[i]
                C12_i = self.mat.C12_prony[i]
                C22_i = self.mat.C22_prony[i]
                C23_i = self.mat.C23_prony[i]
                C66_i = self.mat.C66_prony[i]

                q11_i = q[:, 0:1, i]
                q22_i = q[:, 1:2, i]
                q33_i = q[:, 2:3, i]
                q23_i = q[:, 3:4, i]
                q13_i = q[:, 4:5, i]
                q12_i = q[:, 5:6, i]

                # Add C_i : (eps - q)
                sm_11 += C11_i * (em_mech_11 - q11_i) + C12_i * (em_mech_22 - q22_i) + C12_i * (em_mech_33 - q33_i)
                sm_22 += C12_i * (em_mech_11 - q11_i) + C22_i * (em_mech_22 - q22_i) + C23_i * (em_mech_33 - q33_i)
                sm_33 += C12_i * (em_mech_11 - q11_i) + C23_i * (em_mech_22 - q22_i) + C22_i * (em_mech_33 - q33_i)
                sm_23 += ((C22_i - C23_i) / 2.0) * (gm_mech_23 - q23_i)
                sm_13 += C66_i * (gm_mech_13 - q13_i)
                sm_12 += C66_i * (gm_mech_12 - q12_i)

        sigma_mat_tensor = torch.zeros(batch_size, 3, 3, device=device)
        sigma_mat_tensor[:, 0, 0] = sm_11.squeeze()
        sigma_mat_tensor[:, 1, 1] = sm_22.squeeze()
        sigma_mat_tensor[:, 2, 2] = sm_33.squeeze()
        sigma_mat_tensor[:, 0, 1] = sigma_mat_tensor[:, 1, 0] = sm_12.squeeze()
        sigma_mat_tensor[:, 0, 2] = sigma_mat_tensor[:, 2, 0] = sm_13.squeeze()
        sigma_mat_tensor[:, 1, 2] = sigma_mat_tensor[:, 2, 1] = sm_23.squeeze()

        # sigma_glob = Q^T * sigma_mat * Q
        sigma_glob_tensor = torch.matmul(torch.matmul(Q.T, sigma_mat_tensor), Q)

        sigma_11 = sigma_glob_tensor[:, 0, 0:1]
        sigma_22 = sigma_glob_tensor[:, 1, 1:2]
        sigma_33 = sigma_glob_tensor[:, 2, 2:3]
        sigma_12 = sigma_glob_tensor[:, 0, 1:2]
        sigma_13 = sigma_glob_tensor[:, 0, 2:3]
        sigma_23 = sigma_glob_tensor[:, 1, 2:3]

        # Return: global stresses + material mechanical strains (needed for evolution equation)
        return (sigma_11, sigma_22, sigma_33, sigma_23, sigma_13, sigma_12,
                em_mech_11, em_mech_22, em_mech_33, gm_mech_23, gm_mech_13, gm_mech_12)

    def evolution_residual(self, em_mech_list, q, t, T):
        """
        Compute internal variable evolution residual using DISCRETE time-stepping form
        (safer and more stable than continuous derivative form)

        Physics (Prony discrete update rule):
            q_i^{n+1} = γ_i * q_i^n + (1 - γ_i) * ε_mech^{n+1}
            where γ_i = exp(-Δt / (a_T * τ_i))

        This discrete form:
        - At high T: γ→0, q follows ε (shape recovery)
        - At low T: γ→1, q frozen (shape fixity)
        - NO time derivatives needed (more stable!)
        - Residual has "strain" units instead of "strain rate"

        Args:
            em_mech_list: List of 6 material mechanical strain components (batch, 1)
                         [em_11, em_22, em_33, gm_23, gm_13, gm_12]
            q: Internal variables (batch, 6, N_prony)
            t: Time (batch, 1)
            T: Temperature (batch, 1)

        Returns:
            loss_evol: Evolution equation residual loss
        """
        if q is None or not self.use_physics_loss:
            return torch.tensor(0.0, device=device)

        batch_size = t.shape[0]

        # Use a small time step Δt for discrete approximation
        dt = 1.0  # seconds (typical frame interval in FE data)

        # Current state at t_n
        t_n = t
        T_n = T
        a_T_n = self.mat.shift_factor(T_n)  # (batch, 1)

        # Next state at t_{n+1} = t_n + dt
        t_np1 = t_n + dt
        t_np1 = torch.clamp(t_np1, max=self.bounds['t_max'])  # Don't exceed time domain
        T_np1 = self.get_temperature(t_np1)
        T_np1.requires_grad_(True)

        # We need to evaluate the network at the next time step
        # Reuse input coordinates x, y, z but with t_{n+1}
        # NOTE: This requires x, y, z from the calling context
        # For now, assume we're evaluating at same spatial points
        # (This is handled in pde_residual where this function is called)

        # Since we can't easily get x,y,z here, we use a simpler approach:
        # Assume q is approximately constant over small Δt, and only ε changes
        # This is valid for small time steps and gives similar physics

        eps_mech_np1 = torch.cat(em_mech_list, dim=1)  # (batch, 6)

        # γ_i = exp(-Δt / (a_T * τ_i))
        # Reshape for broadcasting: a_T: (batch,1), tau: (n_prony,)
        tau = torch.tensor(self.mat.rho, dtype=torch.float32, device=device)  # (n_prony,)

        dt_eff = dt / a_T_n  # (batch, 1)
        dt_eff = dt_eff.unsqueeze(-1)  # (batch, 1, 1)
        tau = tau.view(1, 1, -1)  # (1, 1, n_prony)

        # Compute γ for each branch
        gamma = torch.exp(-dt_eff / tau)  # (batch, 1, n_prony)

        # Expand eps_mech for broadcasting with q
        eps_mech_np1_expanded = eps_mech_np1.unsqueeze(-1)  # (batch, 6, 1)

        # Discrete Prony evolution: q^{n+1} ≈ γ * q^n + (1-γ) * ε^{n+1}
        q_pred_np1 = gamma * q + (1.0 - gamma) * eps_mech_np1_expanded  # (batch, 6, n_prony)

        # For numerical stability, we approximate q at t+dt by assuming
        # the network output q is actually q^{n+1}, and we penalize deviation
        # from the predicted value based on current q and eps
        #
        # In practice: residual = q_current - q_predicted
        # But since we only have q at current t, we use a relaxed form:
        # Residual ≈ (q - γ*q - (1-γ)*ε) = q*(1-γ) - (1-γ)*ε = (1-γ)*(q - ε)

        # Simplified stable form: penalize (q - ε) weighted by (1-γ)
        # At high T: (1-γ)→1, strongly enforce q≈ε
        # At low T: (1-γ)→0, weakly enforce (allow q to drift from ε)
        residual = (1.0 - gamma) * (q - eps_mech_np1_expanded)  # (batch, 6, n_prony)

        # Residual has units of strain, normalize by typical strain
        eps0 = 0.1  # Typical strain magnitude

        loss_evol = torch.mean((residual / eps0) ** 2)

        return loss_evol

    def pde_residual(self, x, y, z, t, T):
        """
        Compute PDE residual: ∇·σ + b = 0 (balance of momentum)
        Plus internal variable evolution ODE residual

        Returns:
            f_x, f_y, f_z: Momentum balance residuals (dimensionless)
            loss_evol: Internal variable evolution residual loss
        """
        x.requires_grad_(True)
        y.requires_grad_(True)
        z.requires_grad_(True)
        t.requires_grad_(True)
        T.requires_grad_(True)

        # Forward pass
        outputs = self.model(x, y, z, t, T)
        if self.model.output_internal_vars:
            u_x, u_y, u_z, q = outputs
        else:
            u_x, u_y, u_z = outputs
            q = None

        # Compute strains (global frame)
        eps_11, eps_22, eps_33, gamma_23, gamma_13, gamma_12 = \
            self.compute_strain(u_x, u_y, u_z, x, y, z)

        # Thermal strains (material frame)
        eps_th_11, eps_th_22, eps_th_33, eps_th_23, eps_th_13, eps_th_12 = self.compute_thermal_strain(T)

        # Compute stresses (returns global stresses + material mechanical strains)
        stress_and_strain = self.compute_stress(
            eps_11, eps_22, eps_33, gamma_23, gamma_13, gamma_12,
            eps_th_11, eps_th_22, eps_th_33, eps_th_23, eps_th_13, eps_th_12,
            T, t, q
        )
        sigma_11, sigma_22, sigma_33, sigma_23, sigma_13, sigma_12 = stress_and_strain[:6]
        em_mech_11, em_mech_22, em_mech_33, gm_mech_23, gm_mech_13, gm_mech_12 = stress_and_strain[6:]

        # Stress divergence (global frame)
        sigma_11_x = torch.autograd.grad(sigma_11, x, grad_outputs=torch.ones_like(sigma_11),
                                         create_graph=True, retain_graph=True)[0]
        sigma_12_y = torch.autograd.grad(sigma_12, y, grad_outputs=torch.ones_like(sigma_12),
                                         create_graph=True, retain_graph=True)[0]
        sigma_13_z = torch.autograd.grad(sigma_13, z, grad_outputs=torch.ones_like(sigma_13),
                                         create_graph=True, retain_graph=True)[0]

        sigma_12_x = torch.autograd.grad(sigma_12, x, grad_outputs=torch.ones_like(sigma_12),
                                         create_graph=True, retain_graph=True)[0]
        sigma_22_y = torch.autograd.grad(sigma_22, y, grad_outputs=torch.ones_like(sigma_22),
                                         create_graph=True, retain_graph=True)[0]
        sigma_23_z = torch.autograd.grad(sigma_23, z, grad_outputs=torch.ones_like(sigma_23),
                                         create_graph=True, retain_graph=True)[0]

        sigma_13_x = torch.autograd.grad(sigma_13, x, grad_outputs=torch.ones_like(sigma_13),
                                         create_graph=True, retain_graph=True)[0]
        sigma_23_y = torch.autograd.grad(sigma_23, y, grad_outputs=torch.ones_like(sigma_23),
                                         create_graph=True, retain_graph=True)[0]
        sigma_33_z = torch.autograd.grad(sigma_33, z, grad_outputs=torch.ones_like(sigma_33),
                                         create_graph=True, retain_graph=True)[0]

        # Balance equations (assuming zero body force)
        # ∇·σ = 0
        f_x = sigma_11_x + sigma_12_y + sigma_13_z
        f_y = sigma_12_x + sigma_22_y + sigma_23_z
        f_z = sigma_13_x + sigma_23_y + sigma_33_z

        # Units: Pa/mm -> dimensionless
        stress_grad_ref = self.mat.E_ref / self.mat.L_ref  # Pa/mm
        f_x = f_x / stress_grad_ref
        f_y = f_y / stress_grad_ref
        f_z = f_z / stress_grad_ref

        # Compute evolution residual (material frame)
        em_mech_list = [em_mech_11, em_mech_22, em_mech_33, gm_mech_23, gm_mech_13, gm_mech_12]
        loss_evol = self.evolution_residual(em_mech_list, q, t, T)

        return f_x, f_y, f_z, loss_evol

    def boundary_condition_loss(self, x_bc, y_bc, z_bc, t_bc, T_bc, bc_type, bc_value=None):
        """
        Compute boundary condition loss

        Boundary conditions (45° coupon under X-direction tension):
        - Left face (x=0): Dirichlet BC, u=0 (clamped end, RP1) - enforced here
        - Right face (x=33mm): Loading/displacement BC (driven end, RP2)
          - Implicitly constrained by FE data loss through kinematic coupling
          - Prescribed displacement along X applied via Amplitude=load
          - No explicit Neumann residual needed here

        bc_type: 'dirichlet_left' for left face clamped
        """
        outputs = self.model(x_bc, y_bc, z_bc, t_bc, T_bc)
        if self.model.output_internal_vars:
            u_x, u_y, u_z, q = outputs
        else:
            u_x, u_y, u_z = outputs

        if bc_type == 'dirichlet_left':
            # Left face clamped (u = 0 at x=0, RP1 fully fixed)
            loss = self.normalize_residual(u_x, self.L_ref) + \
                   self.normalize_residual(u_y, self.L_ref) + \
                   self.normalize_residual(u_z, self.L_ref)
        else:
            loss = torch.tensor(0.0, device=device)

        return loss

    def data_loss(self, x_data, y_data, z_data, t_data, T_data, u1_data, u2_data, u3_data):
        """
        Compute data matching loss with FE results
        All displacement components are normalized by L_ref to ensure dimensionless residuals

        Adaptive weighting is applied to balance different displacement magnitudes:
        - U1 (main loading): weight = 1.0
        - U2 (coupling): weight = 5.0 (to compensate ~1/6 magnitude)
        - U3 (Poisson): weight = 20.0 (to compensate ~1/28 magnitude)
        """
        outputs = self.model(x_data, y_data, z_data, t_data, T_data)
        if self.model.output_internal_vars:
            u_x_pred, u_y_pred, u_z_pred, q = outputs
        else:
            u_x_pred, u_y_pred, u_z_pred = outputs

        # Normalize each displacement component by L_ref (mm) to make dimensionless
        loss_u1 = self.normalize_residual(u_x_pred - u1_data, self.L_ref)
        loss_u2 = self.normalize_residual(u_y_pred - u2_data, self.L_ref)
        loss_u3 = self.normalize_residual(u_z_pred - u3_data, self.L_ref)

        # Apply adaptive weights to balance different displacement scales
        return self.lambda_u1 * loss_u1 + self.lambda_u2 * loss_u2 + self.lambda_u3 * loss_u3

    def train_step(self, data_batch, pde_batch, bc_batch):
        """Single training step with FE data"""

        # Data loss (main supervision from FE)
        loss_data = self.data_loss(
            data_batch['x'], data_batch['y'], data_batch['z'], data_batch['t'], data_batch['T'],
            data_batch['u1'], data_batch['u2'], data_batch['u3']
        )

        # PDE residual loss + Evolution loss (physics guidance)
        f_x, f_y, f_z, loss_evol = self.pde_residual(
            pde_batch['x'], pde_batch['y'], pde_batch['z'], pde_batch['t'], pde_batch['T']
        )
        loss_pde = torch.mean(f_x**2 + f_y**2 + f_z**2)

        # Boundary condition loss
        loss_bc = self.boundary_condition_loss(
            bc_batch['x'], bc_batch['y'], bc_batch['z'], bc_batch['t'], bc_batch['T'],
            'dirichlet_left'
        )

        # Total loss (includes evolution ODE constraint)
        loss_total = (self.lambda_data * loss_data +
                     self.lambda_pde * loss_pde +
                     self.lambda_bc * loss_bc +
                     self.lambda_evol * loss_evol)

        return loss_total, loss_data, loss_pde, loss_bc, loss_evol

    def train(self, epochs=5000, batch_size=1024, learning_rate=1e-3,
              n_pde_points=1000, n_bc_points=200):
        """Training loop with FE data"""

        optimizer = Adam(self.model.parameters(), lr=learning_rate)
        scheduler = StepLR(optimizer, step_size=2500, gamma=0.5)

        start_time = time.time()

        # Prepare training data
        full_data = self.fe_loader.full_data
        n_total = len(full_data)

        # Get boundary nodes (left face: x=x_min)
        left_nodes = self.fe_loader.get_boundary_nodes()

        for epoch in range(epochs):
            self.model.train()
            optimizer.zero_grad()

            # Sample data batch from FE results
            indices = np.random.choice(n_total, size=min(batch_size, n_total), replace=False)
            batch_data = full_data.iloc[indices]

            data_batch = {
                'x': torch.tensor(batch_data['X'].values, dtype=torch.float32, device=device).reshape(-1, 1),
                'y': torch.tensor(batch_data['Y'].values, dtype=torch.float32, device=device).reshape(-1, 1),
                'z': torch.tensor(batch_data['Z'].values, dtype=torch.float32, device=device).reshape(-1, 1),
                't': torch.tensor(batch_data['Time'].values, dtype=torch.float32, device=device).reshape(-1, 1),
                'T': torch.tensor(batch_data['Temperature'].values, dtype=torch.float32, device=device).reshape(-1, 1),
                'u1': torch.tensor(batch_data['U1'].values, dtype=torch.float32, device=device).reshape(-1, 1),
                'u2': torch.tensor(batch_data['U2'].values, dtype=torch.float32, device=device).reshape(-1, 1),
                'u3': torch.tensor(batch_data['U3'].values, dtype=torch.float32, device=device).reshape(-1, 1),
            }

            # Sample PDE collocation points
            t_pde = torch.rand(n_pde_points, 1, requires_grad=True, device=device) * (self.bounds['t_max'] - self.bounds['t_min']) + self.bounds['t_min']
            T_pde = self.get_temperature(t_pde)
            T_pde.requires_grad_(True)

            pde_batch = {
                'x': torch.rand(n_pde_points, 1, requires_grad=True, device=device) * (self.bounds['x_max'] - self.bounds['x_min']) + self.bounds['x_min'],
                'y': torch.rand(n_pde_points, 1, requires_grad=True, device=device) * (self.bounds['y_max'] - self.bounds['y_min']) + self.bounds['y_min'],
                'z': torch.rand(n_pde_points, 1, requires_grad=True, device=device) * (self.bounds['z_max'] - self.bounds['z_min']) + self.bounds['z_min'],
                't': t_pde,
                'T': T_pde,
            }

            # Sample boundary points (left face: x=x_min, clamped end)
            bc_indices = np.random.choice(len(left_nodes), size=min(n_bc_points, len(left_nodes)), replace=False)
            bc_data = left_nodes.iloc[bc_indices]

            bc_batch = {
                'x': torch.tensor(bc_data['X'].values, dtype=torch.float32, device=device).reshape(-1, 1),
                'y': torch.tensor(bc_data['Y'].values, dtype=torch.float32, device=device).reshape(-1, 1),
                'z': torch.tensor(bc_data['Z'].values, dtype=torch.float32, device=device).reshape(-1, 1),
                't': torch.tensor(bc_data['Time'].values, dtype=torch.float32, device=device).reshape(-1, 1),
                'T': torch.tensor(bc_data['Temperature'].values, dtype=torch.float32, device=device).reshape(-1, 1),
            }

            # Training step
            loss_total, loss_data, loss_pde, loss_bc, loss_evol = self.train_step(
                data_batch, pde_batch, bc_batch
            )

            loss_total.backward()
            optimizer.step()
            scheduler.step()

            # Store loss history
            self.loss_history.append(loss_total.item())
            self.loss_components_history.append({
                'data': loss_data.item(),
                'pde': loss_pde.item(),
                'bc': loss_bc.item(),
                'evol': loss_evol.item()
            })

            # Print progress
            if epoch % 100 == 0:
                elapsed = time.time() - start_time

    def save_loss_history(self, csv_path):
        """Save recorded loss history to a CSV file"""
        if len(self.loss_history) == 0:
            return

        csv_path = Path(csv_path)
        csv_path.parent.mkdir(parents=True, exist_ok=True)

        history_records = []
        for epoch_idx, loss_total in enumerate(self.loss_history, start=1):
            components = {}
            if epoch_idx - 1 < len(self.loss_components_history):
                components = self.loss_components_history[epoch_idx - 1]

            history_records.append({
                'epoch': epoch_idx,
                'loss_total': loss_total,
                'loss_data': components.get('data', float('nan')),
                'loss_pde': components.get('pde', float('nan')),
                'loss_bc': components.get('bc', float('nan')),
                'loss_evol': components.get('evol', float('nan')),
            })

        df_history = pd.DataFrame(history_records)
        df_history.to_csv(csv_path, index=False)

    def predict(self, x, y, z, t, T=None):
        """
        Make predictions
        Args:
            x, y, z: spatial coordinates
            t: time
            T: temperature (optional, if None will be derived from time)
        """
        self.model.eval()
        with torch.no_grad():
            x_t = torch.tensor(x, dtype=torch.float32, device=device).reshape(-1, 1)
            y_t = torch.tensor(y, dtype=torch.float32, device=device).reshape(-1, 1)
            z_t = torch.tensor(z, dtype=torch.float32, device=device).reshape(-1, 1)
            t_t = torch.tensor(t, dtype=torch.float32, device=device).reshape(-1, 1)

            if T is None:
                # Derive temperature from time if not provided
                T_t = self.get_temperature(t_t)
            else:
                T_t = torch.tensor(T, dtype=torch.float32, device=device).reshape(-1, 1)

            outputs = self.model(x_t, y_t, z_t, t_t, T_t)
            if self.model.output_internal_vars:
                u_x, u_y, u_z, q = outputs
            else:
                u_x, u_y, u_z = outputs

            return u_x.cpu().numpy(), u_y.cpu().numpy(), u_z.cpu().numpy()

## plot_training_history

Implementation of the `plot_training_history` function.

Plot training loss history with Times New Roman font and larger size

In [7]:
def plot_training_history(solver, save_dir=None, prefix=''):
    """Plot training loss history with Times New Roman font and larger size

    Args:
        solver: PINNSolver or LSTM_PINNSolver instance
        save_dir: Directory to save the plot (default: script directory)
        prefix: Filename prefix (e.g., 'lstm_pinn_' or '')
    """
    if save_dir is None:
        save_dir = Path('.')

    # Set font to Times New Roman
    plt.rcParams['font.family'] = 'Times New Roman'
    plt.rcParams['font.size'] = 14
    plt.rcParams['axes.labelsize'] = 16
    plt.rcParams['axes.titlesize'] = 18
    plt.rcParams['legend.fontsize'] = 14

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Total loss
    axes[0].semilogy(solver.loss_history, 'b-', linewidth=2.5)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Total Loss')
    axes[0].set_title('Training Loss History')
    axes[0].grid(True, alpha=0.3, linestyle='--')

    # Loss components (compatible with both dict and list formats)
    if hasattr(solver, 'loss_components_history'):
        # PINN format (list of dicts)
        data_losses = [h['data'] for h in solver.loss_components_history]
        pde_losses = [h['pde'] for h in solver.loss_components_history]
        bc_losses = [h['bc'] for h in solver.loss_components_history]
        evol_losses = [h.get('evol', 0.0) for h in solver.loss_components_history]
    elif hasattr(solver, 'loss_components'):
        # LSTM-PINN format (dict of lists)
        data_losses = solver.loss_components['data']
        pde_losses = solver.loss_components['pde']
        bc_losses = solver.loss_components['bc']
        evol_losses = solver.loss_components.get('evol', [0.0] * len(data_losses))
    else:
        raise AttributeError("Solver must have either 'loss_components_history' or 'loss_components'")

    axes[1].semilogy(data_losses, 'b-', label='Data', linewidth=2.5)
    axes[1].semilogy(pde_losses, 'r-', label='PDE', linewidth=2.5)
    axes[1].semilogy(bc_losses, 'g-', label='BC', linewidth=2.5)
    # Plot evolution loss if it exists and is non-zero
    if any(e > 1e-10 for e in evol_losses):
        axes[1].semilogy(evol_losses, 'm-', label='Evol', linewidth=2.5)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss Components')
    axes[1].set_title('Loss Components History')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, linestyle='--')

    plt.tight_layout()
    save_path = Path(save_dir) / f'{prefix}training_history.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

    # Reset to default
    plt.rcParams.update(plt.rcParamsDefault)

## plot_displacement_field

Implementation of the `plot_displacement_field` function.

Plot 2D displacement field comparison in XY plane for 45° coupon

In [8]:
def plot_displacement_field(solver, fe_loader, save_dir=None, prefix=''):
    """
    Plot 2D displacement field comparison in XY plane for 45° coupon
    Shows FE vs PINN displacement magnitude at selected time points

    Args:
        solver: PINNSolver or LSTM_PINNSolver instance
        fe_loader: FEDataLoader instance
        save_dir: Directory to save the plot (default: script directory)
        prefix: Filename prefix (e.g., 'lstm_pinn_' or '')
    """
    if save_dir is None:
        save_dir = Path('.')

    from matplotlib.tri import Triangulation

    plt.rcParams['font.family'] = 'Times New Roman'
    plt.rcParams['font.size'] = 12
    plt.rcParams['axes.labelsize'] = 14
    plt.rcParams['axes.titlesize'] = 16

    # Select representative time points from the 4-step cycle
    # Step1(0-20s): Loading at high temp (343K)
    # Step2(20-70s): Cooling while loaded
    # Step3(70-71s): Unloading at low temp (298K)
    # Step4(71-121s): Reheating (shape recovery)
    times = [20, 70, 71, 121]  # Key points: end of loading, cooled, unloaded, recovered

    # Get data at midplane (z ≈ 1mm, middle of thickness)
    full_data = fe_loader.full_data
    z_mid = (full_data['Z'].min() + full_data['Z'].max()) / 2
    tolerance = 0.2  # mm

    # Create figure: 3 columns (FE, PINN, Error) × rows (time points)
    n_times = len(times)
    fig, axes = plt.subplots(n_times, 3, figsize=(15, 4*n_times))
    if n_times == 1:
        axes = axes.reshape(1, -1)

    for idx, t_target in enumerate(times):

        # Find closest time frame
        time_mask = np.abs(full_data['Time'] - t_target) < 0.5
        z_mask = np.abs(full_data['Z'] - z_mid) < tolerance
        data_slice = full_data[time_mask & z_mask]

        if len(data_slice) == 0:
            continue

        x = data_slice['X'].values
        y = data_slice['Y'].values
        u1_fe = data_slice['U1'].values
        u2_fe = data_slice['U2'].values
        u3_fe = data_slice['U3'].values
        u_mag_fe = np.sqrt(u1_fe**2 + u2_fe**2 + u3_fe**2)

        # Get PINN predictions
        t = data_slice['Time'].values
        u1_pinn, u2_pinn, u3_pinn = solver.predict(x, y, data_slice['Z'].values, t)
        u_mag_pinn = np.sqrt(u1_pinn.flatten()**2 + u2_pinn.flatten()**2 + u3_pinn.flatten()**2)

        # Compute error
        u_error = np.abs(u_mag_pinn - u_mag_fe)

        # Create triangulation for contour plots
        tri = Triangulation(x, y)

        # Determine common colorbar range
        vmin_disp = min(u_mag_fe.min(), u_mag_pinn.min())
        vmax_disp = max(u_mag_fe.max(), u_mag_pinn.max())
        vmax_error = u_error.max()

        # Determine cycle stage for title
        if t_target <= 20:
            stage = 'Loading (343K)'
        elif t_target <= 70:
            stage = 'Cooling'
        elif t_target <= 71:
            stage = 'Unloading (298K)'
        else:
            stage = 'Recovery (Reheating)'

        # Plot FE
        ax_fe = axes[idx, 0]
        tcf_fe = ax_fe.tricontourf(tri, u_mag_fe, levels=15, cmap='viridis', vmin=vmin_disp, vmax=vmax_disp)
        ax_fe.set_xlabel('X (mm)')
        ax_fe.set_ylabel('Y (mm)')
        ax_fe.set_title(f'FE: {stage} (t={t_target}s)')
        ax_fe.set_aspect('equal')
        fig.colorbar(tcf_fe, ax=ax_fe, label='|u| (mm)')

        # Plot PINN
        ax_pinn = axes[idx, 1]
        tcf_pinn = ax_pinn.tricontourf(tri, u_mag_pinn, levels=15, cmap='viridis', vmin=vmin_disp, vmax=vmax_disp)
        ax_pinn.set_xlabel('X (mm)')
        ax_pinn.set_ylabel('Y (mm)')
        ax_pinn.set_title(f'PINN: {stage} (t={t_target}s)')
        ax_pinn.set_aspect('equal')
        fig.colorbar(tcf_pinn, ax=ax_pinn, label='|u| (mm)')

        # Plot Error
        ax_err = axes[idx, 2]
        tcf_err = ax_err.tricontourf(tri, u_error, levels=15, cmap='hot', vmin=0, vmax=vmax_error)
        ax_err.set_xlabel('X (mm)')
        ax_err.set_ylabel('Y (mm)')
        ax_err.set_title(f'Error: {stage} (t={t_target}s)')
        ax_err.set_aspect('equal')
        fig.colorbar(tcf_err, ax=ax_err, label='|Error| (mm)')

        # Print statistics
        mae = np.mean(u_error)
        rmse = np.sqrt(np.mean(u_error**2))

    plt.tight_layout()
    save_path = Path(save_dir) / f'{prefix}displacement_field_2d.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

    # Reset to default
    plt.rcParams.update(plt.rcParamsDefault)

## plot_shape_memory_cycle

Implementation of the `plot_shape_memory_cycle` function.

Plot shape-memory recovery curve comparing RT-RPINN vs FE for 45° coupon

In [9]:
def plot_shape_memory_cycle(solver, fe_loader, save_dir=None, prefix=''):
    """
    Plot shape-memory recovery curve comparing PINN vs FE for 45° coupon

    Args:
        solver: PINNSolver or LSTM_PINNSolver instance
        fe_loader: FEDataLoader instance
        save_dir: Directory to save the plot (default: script directory)
        prefix: Filename prefix (e.g., 'lstm_pinn_' or '')
    """
    if save_dir is None:
        save_dir = Path('.')

    plt.rcParams['font.family'] = 'Times New Roman'
    plt.rcParams['font.size'] = 12
    plt.rcParams['axes.labelsize'] = 14
    plt.rcParams['axes.titlesize'] = 16
    plt.rcParams['legend.fontsize'] = 12

    n_time_points = 200

    # Monitor point at center of right face (loaded end)
    # Geometry: 33×6×2 mm (L×W×T)
    x = np.ones(n_time_points) * 33.0  # Right face (loaded end)
    y = np.ones(n_time_points) * 3.0   # Width center
    z = np.ones(n_time_points) * 1.0   # Thickness center
    t = np.linspace(0, 121, n_time_points)

    # PINN predictions
    u_x_pinn, u_y_pinn, u_z_pinn = solver.predict(x, y, z, t)

    # Get FE data at monitoring point (use tolerance for floating point comparison)
    tol = 0.1
    fe_monitor = fe_loader.full_data[
        (np.abs(fe_loader.full_data['X'] - 33.0) < tol) &
        (np.abs(fe_loader.full_data['Y'] - 3.0) < tol) &
        (np.abs(fe_loader.full_data['Z'] - 1.0) < tol)
    ]

    # Temperature history based on simulation steps
    T = np.zeros_like(t)
    for i, t_val in enumerate(t):
        if t_val < 20:
            T[i] = 343.0
        elif t_val < 70:
            T[i] = 343.0 - (343.0 - 298.0) * (t_val - 20) / 50
        elif t_val < 71:
            T[i] = 298.0
        else:
            T[i] = 298.0 + (353.0 - 298.0) * (t_val - 71) / 50

    fig, axes = plt.subplots(3, 1, figsize=(12, 10))

    # Displacement vs time (plot U1 - X-direction displacement)
    axes[0].plot(t, u_x_pinn, 'r-', linewidth=2.5, label='PINN')
    if len(fe_monitor) > 0:
        axes[0].scatter(fe_monitor['Time'], fe_monitor['U1'], c='blue', s=40,
                       alpha=0.7, marker='o', edgecolors='darkblue', label='FE', zorder=5)

    # Add vertical lines for key events with clearer labels
    axes[0].axvline(x=20, color='gray', linestyle='--', linewidth=1.5, alpha=0.6)
    axes[0].axvline(x=70, color='gray', linestyle='--', linewidth=1.5, alpha=0.6)
    axes[0].axvline(x=71, color='gray', linestyle='--', linewidth=1.5, alpha=0.6)

    # Add text annotations for stages
    axes[0].text(10, axes[0].get_ylim()[1]*0.95, 'Loading\n(343K)', ha='center', va='top', fontsize=10, color='darkred')
    axes[0].text(45, axes[0].get_ylim()[1]*0.95, 'Cooling\n(343K→298K)', ha='center', va='top', fontsize=10, color='darkblue')
    axes[0].text(70.5, axes[0].get_ylim()[1]*0.95, 'Unload', ha='center', va='top', fontsize=10, color='darkgreen')
    axes[0].text(96, axes[0].get_ylim()[1]*0.95, 'Recovery\n(298K→353K)', ha='center', va='top', fontsize=10, color='darkorange')

    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('Displacement U$_1$ (mm)')
    axes[0].set_title('Shape-Memory Recovery Cycle: X-Direction Displacement')
    axes[0].legend(loc='upper left')
    axes[0].grid(True, alpha=0.3, linestyle='--')

    # Temperature vs time
    axes[1].plot(t, T - 273.15, 'k-', linewidth=2.5)
    axes[1].axvline(x=20, color='gray', linestyle='--', linewidth=1.5, alpha=0.6)
    axes[1].axvline(x=70, color='gray', linestyle='--', linewidth=1.5, alpha=0.6)
    axes[1].axvline(x=71, color='gray', linestyle='--', linewidth=1.5, alpha=0.6)
    axes[1].set_xlabel('Time (s)')
    axes[1].set_ylabel('Temperature (°C)')
    axes[1].set_title('Temperature Loading History')
    axes[1].grid(True, alpha=0.3, linestyle='--')

    # Error analysis
    if len(fe_monitor) > 0:
        # Interpolate PINN predictions to FE time points
        from scipy.interpolate import interp1d
        u_x_pinn_interp = interp1d(t, u_x_pinn.flatten(), kind='linear', fill_value='extrapolate')
        u_x_pinn_at_fe = u_x_pinn_interp(fe_monitor['Time'].values)
        errors = np.abs(u_x_pinn_at_fe - fe_monitor['U1'].values)

        axes[2].plot(fe_monitor['Time'], errors, 'mo-', linewidth=2, markersize=5, label='Absolute Error')
        axes[2].axhline(y=np.mean(errors), color='red', linestyle='--', linewidth=1.5, label=f'Mean Error: {np.mean(errors):.4f} mm')
        axes[2].axvline(x=20, color='gray', linestyle='--', linewidth=1.5, alpha=0.6)
        axes[2].axvline(x=70, color='gray', linestyle='--', linewidth=1.5, alpha=0.6)
        axes[2].axvline(x=71, color='gray', linestyle='--', linewidth=1.5, alpha=0.6)
        axes[2].set_xlabel('Time (s)')
        axes[2].set_ylabel('Absolute Error (mm)')
        axes[2].set_title('Prediction Error Over Shape-Memory Cycle')
        axes[2].legend(loc='upper right')
        axes[2].grid(True, alpha=0.3, linestyle='--')

        # Print statistics

    plt.tight_layout()
    save_path = Path(save_dir) / f'{prefix}shape_memory_cycle.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

    # Reset to default
    plt.rcParams.update(plt.rcParamsDefault)

## main

Implementation of the `main` function.

In [ ]:
def main():

    # Setup paths - use relative path from script location
    script_dir = Path(__file__).parent if '__file__' in globals() else Path('.')
    data_dir = script_dir / 'EX-1-RESULTS'
    step_frame_time_file = script_dir / 'step-frame-time.csv'

    # Load FE data
    fe_loader = FEDataLoader(data_dir, step_frame_time_file)
    full_data = fe_loader.load_all_data()
    bounds = fe_loader.get_domain_bounds()

    # Initialize material parameters
    mat_params = MaterialParameters()

    # Initialize PINN model with internal variables
    # Set output_internal_vars=False for simpler version, True for full physics
    use_internal_vars = True  # Internal variables q enabled for full thermo-viscoelastic physics
    model = SpatiotemporalPINN(layers=[5, 256, 256, 256, 256, 256],
                               n_prony=mat_params.N_prony,
                               output_internal_vars=use_internal_vars)

    # Initialize solver with physics-based normalization
    use_physics_loss = True  # Set to False for pure data-driven training
    solver = PINNSolver(model, mat_params, fe_loader, bounds, use_physics_loss=use_physics_loss)

    # Train the model
    solver.train(epochs=10000, batch_size=2048, learning_rate=1e-3,
                 n_pde_points=2000, n_bc_points=500)
    loss_csv_path = script_dir / 'training_loss_history.csv'
    solver.save_loss_history(loss_csv_path)

    # Generate visualizations
    plot_training_history(solver, save_dir=script_dir)
    plot_displacement_field(solver, fe_loader, save_dir=script_dir)
    plot_shape_memory_cycle(solver, fe_loader, save_dir=script_dir)

    # Save model
    model_path = script_dir / 'pinn_model.pth'
    torch.save(model.state_dict(), model_path)

if __name__ == "__main__":
    main()